# Meet the pandas Series · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 03. Meet the pandas Series

Create a labeled column and explain positions, labels, and arithmetic.

Before you start: Know dictionary keys, list positions, and the difference between = and ==.

Official reference: https://pandas.pydata.org/docs/user_guide/dsintro.html#series

### M03C01. One column, with labels

The setup imports pandas using the short name pd. A Series is a one-dimensional collection of values with an index: labels attached to those values. Without an explicit index, pandas supplies labels starting at 0. A Series also has a dtype describing how its values are stored.

**Read:** What does pd refer to in pd.Series(...) ?

Your explanation: _write here_.

**Predict:** Which index labels will this Series have?

Your explanation: _write here_.

**Explain:** Is the index printed on the left an extra score?

Your explanation: _write here_.



In [ ]:
# Predict before running: Which index labels will this Series have?
scores = pd.Series([88, 72, 91], name="Score")
print(scores)
print("Number of values:", len(scores))
print("Index:", scores.index.tolist())

**Your turn:** Create a Series called hours containing 8, 5, and 9. Give it the name Study_Hours and print its length.

For the check, keep these result names: `hours`.

**Hint:** Change both the list values and the name= argument.

**Watch for this:** Do not call the dtype text a data row; it describes the Series.

In [ ]:
# Change this code to complete the task.
# Question: Create a Series called hours containing 8, 5, and 9. Give it the name Study_Hours and print its length.
scores = pd.Series([88, 72, 91], name="Score")
print(scores)
print("Number of values:", len(scores))
print("Index:", scores.index.tolist())

**Quick check:** A Series is best described as:

1. A complete folder.
2. A Python keyword.
3. One labeled sequence of values.

**Explain what you learned:** I can distinguish Series values, index labels, and the Series name.

### M03C02. Choose labels or positions explicitly

You can supply meaningful labels, such as student IDs, as the Series index. loc selects an index label; iloc selects an integer position. These can identify the same value, but they ask different questions. Using them explicitly avoids confusing a number used as a label with a position.

**Read:** Which expression asks for student S002, and which asks for the second value?

Your explanation: _write here_.

**Predict:** What two values are printed on the last line?

Your explanation: _write here_.

**Explain:** After sorting scores, will position 1 always belong to S002?

Your explanation: _write here_.



In [ ]:
# Predict before running: What two values are printed on the last line?
scores = pd.Series([88, 72, 91], index=["S001", "S002", "S003"], name="Score")
by_label = scores.loc["S002"]
by_position = scores.iloc[1]
print(scores)
print(by_label, by_position)

**Your turn:** Select S003 by label and the first score by position. Store them as by_label and by_position, then print both.

For the check, keep these result names: `by_label`, `by_position`.

**Hint:** Use loc with the ID text, and iloc with 0 for the first position.

**Watch for this:** Prefer .iloc[0] when you mean first position; do not rely on scores[0] with a text index.

In [ ]:
# Change this code to complete the task.
# Question: Select S003 by label and the first score by position. Store them as by_label and by_position, then print both.
scores = pd.Series([88, 72, 91], index=["S001", "S002", "S003"], name="Score")
by_label = scores.loc["S002"]
by_position = scores.iloc[1]
print(scores)
print(by_label, by_position)

**Quick check:** Which selector is explicitly positional?

1. iloc
2. loc
3. name

**Explain what you learned:** I can select a Series value by label or by position.

### M03C03. Do arithmetic on a whole Series

Pandas can apply an arithmetic operation to all values at once. This is called a vectorized operation. scores + 2 creates a new result; it does not replace scores unless you assign the result back. clip() can enforce an upper or lower boundary when that is the stated rule.

**Read:** Does scores + 2 work on one value or every value?

Your explanation: _write here_.

**Predict:** What is the new result for S003, and what remains in scores for S003?

Your explanation: _write here_.

**Explain:** Why might 101 be unacceptable for a score measured out of 100?

Your explanation: _write here_.



In [ ]:
# Predict before running: What is the new result for S003, and what remains in scores for S003?
scores = pd.Series([88, 72, 99], index=["S001", "S002", "S003"], name="Score")
result = scores + 2
print("Original:")
print(scores)
print("After adding two:")
print(result)

**Your turn:** Add 5 to each score, then cap the result at 100. Keep the original scores unchanged.

For the check, keep these result names: `result`, `scores`.

**Hint:** Apply .clip(upper=100) to the calculated Series.

**Watch for this:** Use clipping only when a justified business or teaching rule calls for it.

In [ ]:
# Change this code to complete the task.
# Question: Add 5 to each score, then cap the result at 100. Keep the original scores unchanged.
scores = pd.Series([88, 72, 99], index=["S001", "S002", "S003"], name="Score")
result = scores + 2
print("Original:")
print(scores)
print("After adding two:")
print(result)

**Quick check:** What does result = scores + 2 normally do?

1. Changes the original scores in place.
2. Creates a new Series assigned to result.
3. Adds a new dictionary key.

**Explain what you learned:** I can calculate a new labeled column without writing a loop.

### M03C04. Labels control Series alignment

When combining two Series, pandas matches index labels, even when their order differs. It does not simply pair the first value in one Series with the first value in the other. A label missing from one side normally produces a missing result for arithmetic.

**Read:** Which project value is added to the quiz score for S001?

Your explanation: _write here_.

**Predict:** What total will S003 have?

Your explanation: _write here_.

**Explain:** Why is label alignment useful for student records?

Your explanation: _write here_.



In [ ]:
# Predict before running: What total will S003 have?
quiz = pd.Series({"S001": 17, "S002": 14, "S003": 19})
project = pd.Series({"S003": 46, "S001": 44, "S002": 36})
result = quiz + project
print(result)

**Your turn:** Remove S002 from project, add the two Series again, and print which result is missing. Do not replace the missing project with zero.

For the check, keep these result names: `result`.

**Hint:** Build project with only S003 and S001; use result.isna() to inspect missing results.

**Watch for this:** Equal lengths do not guarantee correct alignment; inspect the actual index labels.

In [ ]:
# Change this code to complete the task.
# Question: Remove S002 from project, add the two Series again, and print which result is missing. Do not replace the missing project with zero.
quiz = pd.Series({"S001": 17, "S002": 14, "S003": 19})
project = pd.Series({"S003": 46, "S001": 44, "S002": 36})
result = quiz + project
print(result)

**Quick check:** How does pandas add two Series with index labels?

1. By matching the labels.
2. Only by their printed order.
3. By sorting their numerical values first.

**Explain what you learned:** I can explain why label alignment can create missing values.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates